# Dataset 6: Friday-Morning EDA (Botnet ARES)
This notebook investigates Botnet Command-and-Control (C2) communication patterns.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

print(f"Loaded Friday Morning: {df.shape[0]:,} rows")
print(f"Class breakdown:\n{df['Label'].value_counts()}")


## 1. Botnet C2 Destination Ports & Flow IAT Periodicity

In [ ]:
for c in [c for c in df.columns if c != 'Label']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

bot_df = df[df['Label'] == 'Bot']
print(f"Top Bot Destination Ports:\n{bot_df['Destination Port'].value_counts().head(3)}")

plt.figure(figsize=(8, 4.5))
df['log_Flow_IAT_Mean'] = np.log10(df['Flow IAT Mean'].clip(lower=1))
sns.boxplot(data=df, x='Label', y='log_Flow_IAT_Mean', palette=['#2b5c8f', '#984ea3'])
plt.title("Friday Morning: Flow IAT Mean (Periodic Beaconing)")
plt.show()


## 2. Key Findings & Preprocessing Recommendations
- Botnet traffic targets Port 8080 (64.1%) with periodic beaconing intervals.
- Flow Inter-Arrival Times (`Flow IAT Mean/Max`) are crucial features for separating bot check-ins from normal web traffic.
